
## Overview

This notebook will show you how to create and query a table or DataFrame that you uploaded to DBFS. [DBFS](https://docs.databricks.com/user-guide/dbfs-databricks-file-system.html) is a Databricks File System that allows you to store data for querying inside of Databricks. This notebook assumes that you have a file already inside of DBFS that you would like to read from.

This notebook is written in **Python** so the default cell type is Python. However, you can use different languages by using the `%LANGUAGE` syntax. Python, Scala, SQL, and R are all supported.

In [0]:
# File location and type
file_location = "/FileStore/tables/pageviews_20250101_000000"
file_type = "parquet"

# CSV options
infer_schema = "false"
first_row_is_header = "false"
delimiter = ","

df = spark.read.text("/FileStore/tables/pageviews_20250101_000000")

from pyspark.sql.functions import split, col

df_parsed = df.withColumn("parts", split(col("value"), " ")) \
    .select(
        col("parts")[0].alias("project"),
        col("parts")[1].alias("page"),
        col("parts")[2].cast("int").alias("views")
    )

df_parsed.show(5)


+-------+--------------------+-----+
|project|                page|views|
+-------+--------------------+-----+
|     ""|Category:Pages_wh...|    2|
|     ""|       Help:Contents|    2|
|     ""|Module:Color_cont...|    3|
|     ""|Module:Message_bo...|    2|
|     ""|Special:MyLanguag...|    1|
+-------+--------------------+-----+
only showing top 5 rows



In [0]:
# Create a view or table

temp_table_name = "pageviews_20250101_000000"

df.createOrReplaceTempView(temp_table_name)

In [0]:
%sql

/* Query the created temp table in a SQL cell */

select * from `pageviews_20250101_000000`

value
""""" Category:Pages_where_node_count_is_exceeded 2 0"
""""" Help:Contents 2 0"
""""" Module:Color_contrast/colors 3 0"
""""" Module:Message_box/ombox.css 2 0"
""""" Special:MyLanguage/Wikifunctions:Status_updates/2024-11-07 1 0"
""""" Special:MyLanguage/Wikifunctions:Status_updates/2024-11-13 1 0"
""""" Special:RecentChanges 10 0"
""""" Special:Search 1 0"
""""" Special:UserLogin 1 0"
""""" Template:Already_done 1 0"


In [0]:
# With this registered as a temp view, it will only be available to this particular notebook. If you'd like other users to be able to query this table, you can also create a table from the DataFrame.
# Once saved, this table will persist across cluster restarts as well as allow various users across different notebooks to query this data.
# To do so, choose your table name and uncomment the bottom line.

permanent_table_name = "pageviews_20250101_000000"

# df.write.format("parquet").saveAsTable(permanent_table_name)

In [0]:
print("Liczba partycji domyślna:", df.rdd.getNumPartitions())


Liczba partycji domyślna: 8


In [0]:
# Przykład: przerepartycjonowanie do 8 partycji
df_repartitioned = df.repartition(6)

print("Nowa liczba partycji:", df_repartitioned.rdd.getNumPartitions())


Nowa liczba partycji: 6


In [0]:
output_path = "/FileStore/tables/pageviews_partitioned_8"

df_repartitioned.write.mode("overwrite").parquet(output_path)


In [0]:
import time
start = time.time()
df_parsed.repartition(10).count()
print("Czas z 10 partycjami:", time.time() - start)

start = time.time()
df_parsed.repartition(50).count()
print("Czas z 50 partycjami:", time.time() - start)

Czas z 10 partycjami: 8.7502601146698
Czas z 50 partycjami: 7.304339170455933
